In [ ]:
"""
app.py
------------------------------------------------
Single-file version: run this and it opens a local web page
(Portfolio Optimizer) in your browser, backed by a Flask server
that pulls live NSE data via yfinance on demand.

Why one process instead of a plain HTML file: browser JavaScript
can't call yfinance or Yahoo Finance directly (Python-only library,
no CORS from a browser). So this script serves the page AND exposes
the /api/fetch endpoint that the page's "Fetch Live Data" button
calls, then auto-opens your browser to it.

Run:
    pip install flask yfinance pandas numpy --break-system-packages
    python app.py

That's it — no separate server.py, no index.html file, no xlsx
uploads. It opens http://localhost:5000 automatically.
"""
import concurrent.futures
import threading
import webbrowser

import numpy as np
import pandas as pd
import yfinance as yf
from flask import Flask, jsonify, request

app = Flask(__name__)

# ---------- Stock universe (same list as the original extract_data.py) ----------
STOCKS = [
    "ABB", "ACC", "ADANIENT", "ADANIGREEN", "ADANIPORTS",
    "AMBUJACEM", "APOLLOHOSP", "ASHOKLEY", "ASIANPAINT", "AUROPHARMA",
    "AXISBANK", "BAJAJ-AUTO", "BAJAJFINSV", "BAJFINANCE", "BANDHANBNK",
    "BANKBARODA", "BEL", "BHARTIARTL", "BHEL", "BOSCHLTD",
    "BPCL", "BRITANNIA", "CANBK", "CIPLA", "COALINDIA",
    "COLPAL", "DABUR", "DIVISLAB", "DLF", "DMART",
    "DRREDDY", "EICHERMOT", "ESCORTS", "ETERNAL", "FEDERALBNK",
    "GAIL", "GODREJCP", "GRASIM", "HAL", "HAVELLS",
    "HCLTECH", "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDUNILVR",
    "ICICIBANK", "ICICIPRULI", "IDFCFIRSTB", "INDHOTEL", "INDIGO",
    "INDUSINDBK", "INFY", "IOC", "IRCTC", "IRFC",
    "ITC", "JSWSTEEL", "KOTAKBANK", "LODHA", "LT",
    "MARICO", "MARUTI", "MOTHERSON", "MPHASIS", "NAUKRI",
    "NESTLEIND", "NHPC", "NTPC", "OFSS", "ONGC",
    "PAGEIND", "PAYTM", "PERSISTENT", "PETRONET", "PIDILITIND",
    "PNB", "POLYCAB", "POWERGRID", "RELIANCE", "RVNL",
    "SAIL", "SBILIFE", "SBIN", "SHRIRAMFIN", "SIEMENS",
    "SUNPHARMA", "TATACONSUM", "TATASTEEL", "TCS", "TECHM",
    "TITAN", "TORNTPHARM", "TRENT", "TVSMOTOR", "UBL",
    "ULTRACEMCO", "UNIONBANK", "VEDL", "WIPRO",
]

MAX_WORKERS = 10  # keep 8-15; too aggressive and Yahoo will temp-rate-limit you

PORT = 5000

INDEX_HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<title>Portfolio Optimizer — NSE Risk/Return Terminal</title>
<script src="https://cdn.tailwindcss.com?plugins=forms,container-queries"></script>
<link href="https://fonts.googleapis.com/css2?family=Material+Symbols+Outlined:wght,FILL@100..700,0..1&display=swap" rel="stylesheet"/>
<link href="https://fonts.googleapis.com/css2?family=Manrope:wght@300;400;500;600;700&display=swap" rel="stylesheet"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/xlsx/0.18.5/xlsx.full.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.9.0/d3.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/plotly.js/1.33.1/plotly.min.js"></script>
<script id="tailwind-config">
      tailwind.config = {
        darkMode: "class",
        theme: {
          extend: {
            "colors": {
                    "secondary-fixed-dim": "#c7c6c6",
                    "on-tertiary": "#ffffff",
                    "on-primary": "#ffffff",
                    "on-secondary-container": "#646464",
                    "surface-container-high": "#e8e8e8",
                    "primary-fixed": "#e2e2e2",
                    "on-primary-fixed": "#1b1b1b",
                    "on-error-container": "#93000a",
                    "on-primary-fixed-variant": "#474747",
                    "inverse-on-surface": "#f0f1f1",
                    "tertiary": "#000000",
                    "background": "#ffffff",
                    "surface-tint": "#5e5e5e",
                    "surface-container-low": "#f3f3f4",
                    "error": "#ba1a1a",
                    "outline": "#e5e5e5",
                    "on-tertiary-fixed": "#1a1c1c",
                    "on-primary-container": "#848484",
                    "secondary": "#5e5e5e",
                    "on-surface": "#1a1c1c",
                    "error-container": "#ffdad6",
                    "tertiary-fixed": "#e2e2e2",
                    "on-surface-variant": "#4c4546",
                    "inverse-surface": "#2f3131",
                    "on-secondary-fixed-variant": "#464747",
                    "secondary-fixed": "#e3e2e2",
                    "on-tertiary-fixed-variant": "#454747",
                    "surface-dim": "#dadada",
                    "on-secondary": "#ffffff",
                    "on-background": "#1a1c1c",
                    "surface": "#ffffff",
                    "on-secondary-fixed": "#1b1c1c",
                    "tertiary-fixed-dim": "#c6c6c7",
                    "surface-bright": "#ffffff",
                    "on-tertiary-container": "#838484",
                    "surface-container": "#f8f8f8",
                    "primary": "#000000",
                    "outline-variant": "#f0f0f0",
                    "primary-container": "#1b1b1b",
                    "surface-container-lowest": "#ffffff",
                    "secondary-container": "#e3e2e2",
                    "surface-container-highest": "#e2e2e2",
                    "inverse-primary": "#c6c6c6",
                    "tertiary-container": "#1a1c1c",
                    "surface-variant": "#fafafa",
                    "primary-fixed-dim": "#c6c6c6",
                    "on-error": "#ffffff"
            },
            "fontFamily": {
                    "sans": ["Manrope", "sans-serif"]
            }
          }
        }
      }
    </script>
<style>
  @keyframes scroll { 0%{ transform:translateX(0);} 100%{ transform:translateX(-50%);} }
  .logo-track{ display:flex; width:200%; animation:scroll 26s linear infinite; }
  .logo-track:hover{ animation-play-state:paused; }

  input[type=range]{ accent-color:#000000; }

  table{ width:100%; border-collapse:collapse; font-family:'Manrope',sans-serif; font-size:12.5px; }
  thead th{ text-align:left; padding:10px 12px; color:#5e5e5e; font-weight:600; font-size:10.5px; text-transform:uppercase; letter-spacing:0.7px; border-bottom:1px solid #e5e5e5; white-space:nowrap; }
  tbody td{ padding:10px 12px; border-bottom:1px solid #f0f0f0; white-space:nowrap; color:#1a1c1c; }
  tbody tr:hover td{ background:#f8f8f8; }
  tbody tr:last-child td{ border-bottom:none; }
  td.stock, th.stock{ font-weight:700; color:#000000; }
  .bar-cell{ display:flex; align-items:center; gap:10px; min-width:160px; }
  .bar-track{ flex:1; height:6px; background:#f0f0f0; border-radius:4px; overflow:hidden; min-width:70px; }
  .bar-fill{ height:100%; background:linear-gradient(90deg,#1a1c1c,#5e5e5e); border-radius:4px; }
  .bar-fill.error{ background:linear-gradient(90deg,#ba1a1a,#e08a85); }
  .heat-cell{ text-align:center; border-radius:2px; }
  ::-webkit-scrollbar{ height:8px; width:8px; } ::-webkit-scrollbar-thumb{ background:#e5e5e5; border-radius:4px; }
  #resultsSection{ display:none; }
</style>
</head>
<body class="bg-background text-on-background font-sans antialiased selection:bg-primary-container selection:text-on-primary">

<!-- TopNavBar -->
<nav class="bg-white/80 backdrop-blur-md border-b border-outline-variant w-full sticky top-0 z-50">
  <div class="flex justify-between items-center w-full px-8 py-4 max-w-[1200px] mx-auto">
    <div class="flex items-center gap-2">
      <span class="w-2 h-2 rounded-full bg-primary"></span>
      <div class="text-xl font-bold tracking-tight text-primary">Optimizer</div>
    </div>
    <div class="hidden md:flex gap-6 items-center">
      <a class="text-primary text-sm font-medium" href="#app">Terminal</a>
      <a class="text-secondary text-sm font-medium hover:text-primary transition-colors duration-200" href="#">Method</a>
      <a class="text-secondary text-sm font-medium hover:text-primary transition-colors duration-200" href="#">Docs</a>
    </div>
    <div class="hidden md:flex gap-4 items-center">
      <span class="text-xs font-mono text-secondary border border-outline-variant rounded-full px-3 py-1.5">NSE · MIN-VARIANCE</span>
    </div>
  </div>
</nav>

<main class="w-full">

  <!-- Hero -->
  <section class="max-w-[1200px] mx-auto px-8 pt-24 pb-12">
    <h1 class="text-4xl md:text-5xl font-light text-primary max-w-3xl leading-tight tracking-tight">Precision portfolio construction from raw NSE price data.</h1>
    <p class="text-base text-secondary leading-relaxed font-light max-w-2xl mt-6">
      Pull live NSE prices straight from Yahoo Finance, set a target risk/return quadrant, and let a projected-gradient minimum-variance solver find the allocation — with the Delaunay mesh, convex hull and correlation structure laid bare.
    </p>
  </section>

  <!-- Stock ticker marquee (replaces logo roll) -->
  <section class="w-full overflow-hidden border-y border-outline-variant py-5 mb-16 bg-surface-variant">
    <div class="logo-track flex items-center gap-12 opacity-50 px-8" id="tickerTrack">
      <!-- populated by JS: default list, replaced with uploaded universe once risk.xlsx loads -->
    </div>
  </section>

  <!-- ============ APP ============ -->
  <section id="app" class="max-w-[1200px] mx-auto px-8 pb-32">
    <div class="grid grid-cols-1 lg:grid-cols-[340px_1fr] gap-8 items-start">

      <!-- LEFT: CONTROLS -->
      <div>
        <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6">
          <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-5 flex items-center gap-2">
            <span class="font-mono text-primary">01</span> Data Source — Live (Yahoo Finance)
          </h2>

          <label class="text-xs font-semibold text-secondary uppercase tracking-wider block mb-2">Years of History</label>
          <input type="number" id="fetchYears" value="3" min="1" step="1" class="w-full bg-surface-container-low border border-outline-variant rounded-lg px-3 py-2.5 text-sm font-mono text-on-surface focus:outline-none focus:border-primary transition-colors" />

          <label class="text-xs font-semibold text-secondary uppercase tracking-wider block mb-2 mt-4">VaR95 Period</label>
          <select id="fetchVarPeriod" class="w-full bg-surface-container-low border border-outline-variant rounded-lg px-3 py-2.5 text-sm font-mono text-on-surface focus:outline-none focus:border-primary transition-colors">
            <option value="monthly">Monthly</option>
            <option value="quarterly">Quarterly</option>
          </select>

          <label class="text-xs font-semibold text-secondary uppercase tracking-wider block mb-2 mt-4">Expected Return Period</label>
          <select id="fetchReturnPeriod" class="w-full bg-surface-container-low border border-outline-variant rounded-lg px-3 py-2.5 text-sm font-mono text-on-surface focus:outline-none focus:border-primary transition-colors">
            <option value="monthly">Monthly</option>
            <option value="quarterly">Quarterly</option>
          </select>

          <button id="fetchBtn" class="w-full mt-5 bg-primary text-on-primary font-semibold text-sm uppercase tracking-wide rounded-full py-3.5 hover:bg-primary/90 transition-colors flex items-center justify-center gap-2">
            <span class="material-symbols-outlined text-lg" id="fetchIcon">cloud_download</span>
            <span id="fetchBtnLabel">Fetch Live Data</span>
          </button>
          <div class="text-[10.5px] text-secondary font-mono text-center mt-2" id="fetchStatus">requires local data server — see below</div>
          <div class="text-[10px] text-secondary font-mono mt-3 leading-relaxed border-t border-outline-variant pt-3">
            Run <code class="bg-surface-container-low px-1 py-0.5 rounded">python server.py</code> once (installs/uses <code class="bg-surface-container-low px-1 py-0.5 rounded">yfinance</code>) then click Fetch. Browsers can't call Yahoo Finance directly — this hits <code class="bg-surface-container-low px-1 py-0.5 rounded">localhost:5000</code> instead.
          </div>
        </div>

        <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6 mt-5">
          <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-5 flex items-center gap-2">
            <span class="font-mono text-primary">02</span> Target Quadrant
          </h2>

          <label class="text-xs font-semibold text-secondary uppercase tracking-wider block mb-2">Target Risk (VaR95, %)</label>
          <input type="number" id="userRisk" placeholder="e.g. 5" step="0.1" class="w-full bg-surface-container-low border border-outline-variant rounded-lg px-3 py-2.5 text-sm font-mono text-on-surface focus:outline-none focus:border-primary transition-colors" />
          <div class="text-[10.5px] text-secondary font-mono mt-1.5" id="riskPeriodHint">enter positive value — sign handled internally</div>

          <label class="text-xs font-semibold text-secondary uppercase tracking-wider block mb-2 mt-5">Target Return (%)</label>
          <input type="number" id="userReturn" placeholder="e.g. 3" step="0.1" class="w-full bg-surface-container-low border border-outline-variant rounded-lg px-3 py-2.5 text-sm font-mono text-on-surface focus:outline-none focus:border-primary transition-colors" />
          <div class="text-[10.5px] text-secondary font-mono mt-1.5" id="retPeriodHint">expected average period return</div>

          <label class="text-sm font-medium text-primary block mb-2 mt-5">Closest Stocks to Highlight</label>
          <div class="flex items-center gap-3">
            <input type="range" id="topN" min="2" max="15" value="5" class="flex-1" />
            <div class="font-mono text-sm font-bold text-primary min-w-[24px] text-center" id="topNVal">5</div>
          </div>
          <div class="text-[10.5px] text-secondary font-mono mt-1.5">convex hull needs 3+ stocks to render</div>

          <label class="text-xs font-semibold text-secondary uppercase tracking-wider block mb-2 mt-5">Investment Amount (₹)</label>
          <input type="number" id="investAmount" placeholder="e.g. 50000" min="10000" step="1000" class="w-full bg-surface-container-low border border-outline-variant rounded-lg px-3 py-2.5 text-sm font-mono text-on-surface focus:outline-none focus:border-primary transition-colors" />
          <div class="text-[11.5px] font-mono text-error mt-1.5 hidden" id="investErr">Minimum investment is ₹10,000</div>

          <button id="runBtn" disabled class="w-full mt-6 bg-primary text-on-primary disabled:bg-surface-container-highest disabled:text-secondary font-semibold text-sm uppercase tracking-wide rounded-full py-3.5 hover:bg-primary/90 transition-colors disabled:cursor-not-allowed">Run Optimization</button>
          <div class="text-[10.5px] text-secondary font-mono text-center mt-2" id="runNote">fetch live data to continue</div>
        </div>
      </div>

      <!-- RIGHT: RESULTS -->
      <div>
        <div class="border border-dashed border-outline rounded-2xl flex flex-col items-center justify-center text-center px-10 py-24" id="emptyState">
          <span class="material-symbols-outlined text-secondary text-4xl mb-4">insights</span>
          <div class="text-base font-semibold text-on-surface mb-2">Waiting for inputs</div>
          <div class="text-sm text-secondary max-w-sm leading-relaxed font-light">Click "Fetch Live Data" to pull fresh NSE prices via the local data server, set your target risk/return quadrant, and run the optimizer to see the mesh, top matches and allocation.</div>
        </div>

        <div id="resultsSection">

          <div class="grid grid-cols-2 md:grid-cols-4 gap-4 mb-5">
            <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-5">
              <div class="text-[10.5px] uppercase tracking-wider text-secondary font-semibold">Stocks Selected</div>
              <div class="font-mono text-2xl font-bold mt-1.5 text-primary" id="kpiCount">—</div>
              <div class="text-[10.5px] text-secondary font-mono mt-0.5" id="kpiCountSub">of universe</div>
            </div>
            <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-5">
              <div class="text-[10.5px] uppercase tracking-wider text-secondary font-semibold">Weighted Risk</div>
              <div class="font-mono text-2xl font-bold mt-1.5 text-on-surface" id="kpiRisk">—</div>
              <div class="text-[10.5px] text-secondary font-mono mt-0.5">portfolio VaR95</div>
            </div>
            <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-5">
              <div class="text-[10.5px] uppercase tracking-wider text-secondary font-semibold">Weighted Return</div>
              <div class="font-mono text-2xl font-bold mt-1.5 text-error" id="kpiReturn">—</div>
              <div class="text-[10.5px] text-secondary font-mono mt-0.5">expected period return</div>
            </div>
            <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-5">
              <div class="text-[10.5px] uppercase tracking-wider text-secondary font-semibold">Total Deployed</div>
              <div class="font-mono text-2xl font-bold mt-1.5 text-on-surface" id="kpiInvest">—</div>
              <div class="text-[10.5px] text-secondary font-mono mt-0.5" id="kpiInvestSub">after round-up</div>
            </div>
          </div>

          <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6">
            <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-4 flex items-center gap-2"><span class="font-mono text-primary">03</span> Risk / Return Quadrant</h2>
            <div class="flex gap-5 flex-wrap mb-3 text-[11px] font-mono text-secondary">
              <div class="flex items-center gap-1.5"><span class="w-2.5 h-2.5 rounded-sm" style="background:#5e5e5e"></span>All stocks</div>
              <div class="flex items-center gap-1.5"><span class="w-2.5 h-2.5 rounded-sm" style="background:#000000"></span>Top matches</div>
              <div class="flex items-center gap-1.5"><span class="w-2.5 h-2.5 rounded-sm" style="background:#ba1a1a"></span>Target</div>
              <div class="flex items-center gap-1.5"><span class="w-2.5 h-2.5 rounded-sm" style="background:#5e5e5e"></span>Weighted centroid</div>
            </div>
            <div id="chart" style="width:100%; height:560px;"></div>
          </div>

          <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6 mt-5">
            <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-4 flex items-center gap-2"><span class="font-mono text-primary">04</span> Top Matches</h2>
            <div class="overflow-x-auto"><table id="topTable"></table></div>
          </div>

          <div class="grid grid-cols-1 md:grid-cols-2 gap-5 mt-5">
            <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6">
              <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-4 flex items-center gap-2"><span class="font-mono text-primary">05</span> Correlation Matrix <span class="text-[10px] font-mono border border-outline-variant rounded-full px-2 py-0.5 text-secondary normal-case tracking-normal" id="corrTag"></span></h2>
              <div class="overflow-x-auto"><table id="corrTable"></table></div>
            </div>
            <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6">
              <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-4 flex items-center gap-2"><span class="font-mono text-primary">06</span> Covariance Matrix <span class="text-[10px] font-mono border border-outline-variant rounded-full px-2 py-0.5 text-secondary normal-case tracking-normal" id="covTag"></span></h2>
              <div class="overflow-x-auto"><table id="covTable"></table></div>
            </div>
          </div>

          <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6 mt-5">
            <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-4 flex items-center gap-2"><span class="font-mono text-primary">07</span> Optimal Weights — Minimum Variance</h2>
            <div class="overflow-x-auto"><table id="weightsTable"></table></div>
          </div>

          <div class="bg-surface-container-lowest border border-outline-variant rounded-2xl p-6 mt-5">
            <h2 class="text-xs font-semibold uppercase tracking-wider text-secondary mb-4 flex items-center gap-2"><span class="font-mono text-primary">08</span> Investment Allocation</h2>
            <div class="overflow-x-auto"><table id="allocTable"></table></div>
            <div class="text-[11px] text-secondary font-mono mt-3" id="allocNote"></div>
          </div>

        </div>
      </div>
    </div>
  </section>
</main>

<footer class="bg-background text-primary border-t border-outline-variant w-full">
  <div class="flex flex-col md:flex-row justify-between items-center w-full px-8 py-8 max-w-[1200px] mx-auto gap-4">
    <div class="text-base font-bold text-primary tracking-tight">Optimizer</div>
    <div class="text-xs text-secondary font-light">Minimum-variance allocation, computed client-side.</div>
  </div>
</footer>

<script>
/* =========================================================================
   TICKER MARQUEE
========================================================================= */
const DEFAULT_TICKERS = ["RELIANCE","TCS","HDFC BANK","INFOSYS","ICICI BANK","ITC","SBI","BHARTI AIRTEL","LT","HCL TECH","ASIAN PAINTS","MARUTI SUZUKI"];
function renderTicker(names){
  const track = document.getElementById('tickerTrack');
  const list = names && names.length ? names : DEFAULT_TICKERS;
  const doubled = [...list, ...list];
  track.innerHTML = doubled.map(n => `<div class="text-xl text-primary font-semibold tracking-tight whitespace-nowrap">${n}</div>`).join('');
}
renderTicker(DEFAULT_TICKERS);

/* =========================================================================
   STATE
========================================================================= */
const state = { riskRows:null, stockRows:null, varPeriod:'', returnPeriod:'' };
const DATA_SERVER_URL = '/api/fetch';

/* =========================================================================
   FIELD HELPER (still used by the pipeline below on the fetched JSON rows)
========================================================================= */
function fieldGet(row, names){
  for(const n of names){
    if(row[n] !== undefined) return row[n];
    const key = Object.keys(row).find(k => k.trim().toLowerCase() === n.toLowerCase());
    if(key) return row[key];
  }
  return undefined;
}

/* =========================================================================
   LIVE FETCH (replaces the old xlsx upload flow)
   Talks to the local Flask server in server.py, which wraps yfinance —
   a browser can't call Yahoo Finance directly (no CORS, no JS client).
========================================================================= */
function setFetchStatus(text, isError){
  const el = document.getElementById('fetchStatus');
  el.textContent = text;
  el.style.color = isError ? '#ba1a1a' : '';
}

document.getElementById('fetchBtn').addEventListener('click', async ()=>{
  const years = parseInt(document.getElementById('fetchYears').value, 10) || 3;
  const varPeriod = document.getElementById('fetchVarPeriod').value;
  const returnPeriod = document.getElementById('fetchReturnPeriod').value;

  const btn = document.getElementById('fetchBtn');
  const label = document.getElementById('fetchBtnLabel');
  const icon = document.getElementById('fetchIcon');
  btn.disabled = true;
  label.textContent = 'Fetching...';
  icon.textContent = 'sync';
  icon.classList.add('animate-spin');
  setFetchStatus('contacting local data server…');

  try{
    const res = await fetch(DATA_SERVER_URL, {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ years, varPeriod, returnPeriod })
    });
    if(!res.ok) throw new Error(`server returned ${res.status}`);
    const payload = await res.json();

    if(!payload.riskRows || !payload.riskRows.length){
      throw new Error('no risk data returned — check the server console for failed tickers');
    }

    state.riskRows = payload.riskRows;
    state.stockRows = payload.stockRows;
    state.varPeriod = varPeriod;
    state.returnPeriod = returnPeriod;

    const capVar = varPeriod[0].toUpperCase()+varPeriod.slice(1);
    const capRet = returnPeriod[0].toUpperCase()+returnPeriod.slice(1);
    document.getElementById('riskPeriodHint').textContent = `${capVar} VaR95% — sign handled internally`;
    document.getElementById('retPeriodHint').textContent = `expected average ${capRet.toLowerCase()} return`;

    const names = [...new Set(payload.riskRows.map(r => r.stock).filter(Boolean))];
    if(names.length) renderTicker(names);

    setFetchStatus(`loaded ${payload.riskRows.length} stocks · ${payload.stockRows.length} price rows`);
    checkReady();
  }catch(err){
    setFetchStatus('fetch failed: ' + err.message, true);
    alert('Could not fetch live data.\n\n' + err.message + '\n\nMake sure server.py is running (python server.py) on port 5000.');
  }finally{
    btn.disabled = false;
    label.textContent = 'Fetch Live Data';
    icon.textContent = 'cloud_download';
    icon.classList.remove('animate-spin');
  }
});

function checkReady(){
  const ready = !!state.riskRows && !!state.stockRows;
  document.getElementById('runBtn').disabled = !ready;
  document.getElementById('runNote').textContent = ready ? 'inputs ready' : 'fetch live data to continue';
}

document.getElementById('topN').addEventListener('input', (e)=>{
  document.getElementById('topNVal').textContent = e.target.value;
});

/* =========================================================================
   MATH HELPERS
========================================================================= */
function mean(a){ return a.reduce((s,x)=>s+x,0)/a.length; }
function sampleStd(a){
  const m = mean(a);
  const v = a.reduce((s,x)=>s+(x-m)*(x-m),0)/(a.length-1);
  return Math.sqrt(v);
}
function pearson(a,b){
  const ma=mean(a), mb=mean(b);
  let num=0, da=0, db=0;
  for(let i=0;i<a.length;i++){ num += (a[i]-ma)*(b[i]-mb); da += (a[i]-ma)**2; db += (b[i]-mb)**2; }
  return num / Math.sqrt(da*db);
}
function matVec(M, v){ return M.map(row => row.reduce((s,x,i)=>s+x*v[i], 0)); }

function projectSimplex(v){
  const n = v.length;
  const u = [...v].sort((a,b)=>b-a);
  let cssv = 0, rho = -1, cssvAtRho = 0;
  for(let i=0;i<n;i++){
    cssv += u[i];
    if (u[i] - (cssv-1)/(i+1) > 0){ rho = i; cssvAtRho = cssv; }
  }
  const theta = (cssvAtRho - 1) / (rho + 1);
  return v.map(x => Math.max(x - theta, 0));
}

function minVariancePortfolio(cov){
  const n = cov.length;
  if(n === 1) return [1];
  let b = new Array(n).fill(1);
  for(let i=0;i<60;i++){
    const nb = matVec(cov, b);
    const norm = Math.sqrt(nb.reduce((s,x)=>s+x*x,0)) || 1;
    b = nb.map(x=>x/norm);
  }
  const eigApprox = Math.max(...matVec(cov,b).map((x,i)=>Math.abs(x/(b[i]||1e-9))));
  const L = Math.max(2*eigApprox, 1e-6);
  const step = 1/L;

  let w = new Array(n).fill(1/n);
  for(let iter=0; iter<3000; iter++){
    const grad = matVec(cov, w).map(x=>2*x);
    w = projectSimplex(w.map((wi,i)=> wi - step*grad[i]));
  }
  return w;
}

/* =========================================================================
   FORMATTERS
========================================================================= */
const fmt2 = x => (x===null||x===undefined||isNaN(x)) ? '—' : Number(x).toFixed(2);
const fmt4 = x => (x===null||x===undefined||isNaN(x)) ? '—' : Number(x).toFixed(4);
const fmtRupee = x => '₹' + Number(x).toLocaleString('en-IN');

function heatColor(v, absMax){
  // grayscale (positive) vs error red (negative) — monochrome system, single accent
  const t = Math.max(-1, Math.min(1, v/(absMax||1)));
  if(t >= 0){
    const a = 0.06 + 0.30*t;
    return `background:rgba(0,0,0,${a.toFixed(2)})`;
  } else {
    const a = 0.06 + 0.30*(-t);
    return `background:rgba(186,26,26,${a.toFixed(2)})`;
  }
}

/* =========================================================================
   MAIN PIPELINE
========================================================================= */
document.getElementById('investAmount').addEventListener('input', validateInvest);
function validateInvest(){
  const v = parseFloat(document.getElementById('investAmount').value);
  const bad = isNaN(v) || v < 10000;
  document.getElementById('investErr').classList.toggle('hidden', !(document.getElementById('investAmount').value !== '' && bad));
  return !bad;
}

document.getElementById('runBtn').addEventListener('click', runPipeline);

function runPipeline(){
  const userRiskInput = parseFloat(document.getElementById('userRisk').value);
  const userReturn = parseFloat(document.getElementById('userReturn').value);
  const topN = parseInt(document.getElementById('topN').value, 10);
  const investAmount = parseFloat(document.getElementById('investAmount').value);

  if(isNaN(userRiskInput) || isNaN(userReturn)){ alert('Enter both target risk and target return.'); return; }
  if(!validateInvest()){ document.getElementById('investAmount').focus(); return; }

  const userRisk = -1 * Math.abs(userRiskInput);

  const risk = state.riskRows.map(r => ({
    stock: fieldGet(r,['stock']),
    var95: parseFloat(fieldGet(r,['var95'])),
    avg_ret: parseFloat(fieldGet(r,['avg_ret'])),
  })).filter(r => r.stock != null && !isNaN(r.var95) && !isNaN(r.avg_ret));

  risk.forEach(r => {
    const projX = Math.max(r.var95, userRisk);
    const projY = Math.max(r.avg_ret, userReturn);
    r.dist = Math.hypot(r.var95 - projX, r.avg_ret - projY);
  });

  const sorted = [...risk].sort((a,b)=> a.dist - b.dist);
  const topStocks = sorted.slice(0, Math.min(topN, sorted.length));
  const best = topStocks.map(r=>r.stock);

  const allPoints = risk.map(r=>[r.var95, r.avg_ret]);
  let meshEdges = [];
  if(allPoints.length >= 3){
    const delaunay = d3.Delaunay.from(allPoints);
    const seen = new Set();
    for(let i=0;i<delaunay.triangles.length;i+=3){
      const t = [delaunay.triangles[i], delaunay.triangles[i+1], delaunay.triangles[i+2]];
      for(let k=0;k<3;k++){
        const a=t[k], b=t[(k+1)%3];
        const key = a<b ? a+'_'+b : b+'_'+a;
        if(!seen.has(key)){ seen.add(key); meshEdges.push([allPoints[a], allPoints[b]]); }
      }
    }
  }

  const stockRows = state.stockRows
    .map(r => ({ stock: fieldGet(r,['stock']), date: fieldGet(r,['Date','date']), close: parseFloat(fieldGet(r,['Close','close'])) }))
    .filter(r => r.stock && best.includes(r.stock) && r.date && !isNaN(r.close));

  const isQuarterly = (state.returnPeriod || '').toLowerCase() === 'quarterly';
  function periodKey(d){
    const dt = (d instanceof Date) ? d : new Date(d);
    const y = dt.getFullYear();
    if(isQuarterly){ return `${y}-Q${Math.floor(dt.getMonth()/3)+1}`; }
    return `${y}-${String(dt.getMonth()+1).padStart(2,'0')}`;
  }
  function periodSortKey(d){
    const dt = (d instanceof Date) ? d : new Date(d);
    return dt.getFullYear()*100 + (isQuarterly ? Math.floor(dt.getMonth()/3)+1 : dt.getMonth()+1);
  }

  const lastByPeriodStock = {};
  stockRows.forEach(r=>{
    const pk = periodKey(r.date);
    const sk = periodSortKey(r.date);
    const key = pk+'|'+r.stock;
    const t = (r.date instanceof Date ? r.date.getTime() : new Date(r.date).getTime());
    if(!lastByPeriodStock[key] || t > lastByPeriodStock[key].t){
      lastByPeriodStock[key] = { sortKey:sk, period:pk, stock:r.stock, close:r.close, t };
    }
  });

  const periodsSet = new Set(Object.values(lastByPeriodStock).map(v=>v.period));
  const periods = [...periodsSet].sort((a,b)=>{
    const va = Object.values(lastByPeriodStock).find(v=>v.period===a).sortKey;
    const vb = Object.values(lastByPeriodStock).find(v=>v.period===b).sortKey;
    return va-vb;
  });

  const closeMatrix = periods.map(p => best.map(s => {
    const rec = lastByPeriodStock[p+'|'+s];
    return rec ? rec.close : null;
  }));

  const returnsMatrix = [];
  for(let i=1;i<closeMatrix.length;i++){
    const row = [];
    let ok = true;
    for(let j=0;j<best.length;j++){
      const prev = closeMatrix[i-1][j], curr = closeMatrix[i][j];
      if(prev == null || curr == null || prev === 0){ ok = false; break; }
      row.push((curr-prev)/prev);
    }
    if(ok) returnsMatrix.push(row);
  }

  let corrMatrix = [], covMatrix = [], stds = [], optWeights = [];
  const n = best.length;
  const enoughData = returnsMatrix.length >= 2 && n >= 1;

  if(enoughData){
    const cols = best.map((_,j)=> returnsMatrix.map(r=>r[j]));
    stds = cols.map(c => sampleStd(c));
    corrMatrix = cols.map((ci,i)=> cols.map((cj,j)=> i===j ? 1 : pearson(ci,cj)));
    covMatrix = corrMatrix.map((row,i)=> row.map((c,j)=> c * stds[i] * stds[j]));
    optWeights = n === 1 ? [1] : minVariancePortfolio(covMatrix);
  } else {
    optWeights = best.map(()=> 1/n);
  }

  const topPoints = topStocks.map(r=>[r.var95, r.avg_ret]);
  const weightedRisk = topPoints.reduce((s,p,i)=> s + p[0]*optWeights[i], 0);
  const weightedReturn = topPoints.reduce((s,p,i)=> s + p[1]*optWeights[i], 0);

  let hull = null;
  if(topPoints.length >= 3){
    hull = d3.polygonHull(topPoints);
  }

  const allocation = best.map((s,i)=> ({ stock:s, weight: optWeights[i], amount: Math.ceil(optWeights[i]*investAmount) }));

  renderAll({
    varLabel: labelFor(state.varPeriod, 'VaR95 %', 'Risk'),
    retLabel: labelFor(state.returnPeriod, 'Average Return (%)', 'Return', true),
    risk, topStocks, userRisk, userReturn, meshEdges, hull, topPoints,
    weightedRisk, weightedReturn, corrMatrix, covMatrix, best, optWeights, allocation, investAmount,
    enoughData, periods: periods.length
  });
}

function labelFor(period, fallback, kind, isReturn){
  const cap = period ? period[0].toUpperCase()+period.slice(1) : '';
  if(kind==='Risk') return cap ? `Risk (${cap} VaR95 %)` : 'Risk (VaR95 %)';
  return cap ? `Avg ${cap} Return (%)` : fallback;
}

/* =========================================================================
   RENDERING
========================================================================= */
function renderAll(d){
  document.getElementById('emptyState').style.display = 'none';
  document.getElementById('resultsSection').style.display = 'block';

  document.getElementById('kpiCount').textContent = d.topStocks.length;
  document.getElementById('kpiCountSub').textContent = `of ${d.risk.length} in universe`;
  document.getElementById('kpiRisk').textContent = fmt2(d.weightedRisk);
  document.getElementById('kpiReturn').textContent = fmt2(d.weightedReturn) + '%';
  const totalAlloc = d.allocation.reduce((s,a)=>s+a.amount,0);
  document.getElementById('kpiInvest').textContent = fmtRupee(totalAlloc);
  document.getElementById('kpiInvestSub').textContent = `of ${fmtRupee(d.investAmount)} requested`;

  renderChart(d);
  renderTopTable(d);
  renderCorrCov(d);
  renderWeights(d);
  renderAllocation(d, totalAlloc);
}

function renderChart(d){
  const traces = [];

  if(d.meshEdges.length){
    const mx=[], my=[];
    d.meshEdges.forEach(([a,b])=>{ mx.push(a[0], b[0], null); my.push(a[1], b[1], null); });
    traces.push({ x:mx, y:my, mode:'lines', line:{color:'rgba(0,0,0,0.08)', width:1}, hoverinfo:'skip', showlegend:false, type:'scattergl' });
  }

  const others = d.risk.filter(r => !d.topStocks.includes(r));
  traces.push({
    x: others.map(r=>r.var95), y: others.map(r=>r.avg_ret), text: others.map(r=>r.stock),
    mode:'markers+text', textposition:'top center', textfont:{size:8, color:'#5e5e5e', family:'Manrope'},
    marker:{ size:7, color:'rgba(94,94,94,0.45)', line:{color:'#ffffff', width:1} },
    hovertemplate:'<b>%{text}</b><br>risk %{x:.2f}<br>return %{y:.2f}<extra></extra>',
    name:'All stocks', showlegend:false, type:'scattergl'
  });

  if(d.hull && d.hull.length >= 3){
    const hx = d.hull.map(p=>p[0]).concat([d.hull[0][0]]);
    const hy = d.hull.map(p=>p[1]).concat([d.hull[0][1]]);
    traces.push({ x:hx, y:hy, mode:'lines', fill:'toself', fillcolor:'rgba(0,0,0,0.04)',
      line:{color:'#000000', width:1.3}, hoverinfo:'skip', showlegend:false, type:'scatter' });
  }

  d.topStocks.forEach(r=>{
    traces.push({ x:[d.userRisk, r.var95], y:[d.userReturn, r.avg_ret], mode:'lines',
      line:{color:'rgba(0,0,0,0.22)', width:1, dash:'dot'}, hoverinfo:'skip', showlegend:false, type:'scatter' });
    traces.push({ x:[r.var95, d.weightedRisk], y:[r.avg_ret, d.weightedReturn], mode:'lines',
      line:{color:'rgba(94,94,94,0.30)', width:1, dash:'dot'}, hoverinfo:'skip', showlegend:false, type:'scatter' });
  });

  traces.push({
    x: d.topStocks.map(r=>r.var95), y: d.topStocks.map(r=>r.avg_ret), text: d.topStocks.map(r=>r.stock),
    mode:'markers+text', textposition:'top center', textfont:{size:11, color:'#000000', family:'Manrope', weight:700},
    marker:{ size:13, color:'#000000', line:{color:'#ffffff', width:2} },
    hovertemplate:'<b>%{text}</b><br>risk %{x:.3f}<br>return %{y:.3f}<extra></extra>',
    name:'Top matches', showlegend:false, type:'scatter'
  });

  traces.push({
    x:[d.userRisk], y:[d.userReturn], mode:'markers+text', text:['Target'], textposition:'bottom center',
    textfont:{size:10, color:'#ba1a1a', family:'Manrope'},
    marker:{ symbol:'star', size:18, color:'#ba1a1a', line:{color:'#ffffff', width:1.5} },
    hovertemplate:'<b>Target</b><br>risk %{x:.2f}<br>return %{y:.2f}<extra></extra>',
    name:'Target', showlegend:false, type:'scatter'
  });

  traces.push({
    x:[d.weightedRisk], y:[d.weightedReturn], mode:'markers+text', text:['Centroid'], textposition:'bottom center',
    textfont:{size:10, color:'#5e5e5e', family:'Manrope'},
    marker:{ size:15, color:'#5e5e5e', line:{color:'#ffffff', width:2} },
    hovertemplate:'<b>Weighted Centroid</b><br>risk %{x:.3f}<br>return %{y:.3f}<extra></extra>',
    name:'Centroid', showlegend:false, type:'scatter'
  });

  const layout = {
    paper_bgcolor:'transparent', plot_bgcolor:'transparent',
    font:{ family:'Manrope', color:'#4c4546', size:11 },
    margin:{ l:60, r:30, t:10, b:55 },
    xaxis:{ title:{text:d.varLabel, font:{color:'#4c4546', size:12}}, autorange:'reversed', gridcolor:'rgba(0,0,0,0.06)', zerolinecolor:'rgba(0,0,0,0.12)' },
    yaxis:{ title:{text:d.retLabel, font:{color:'#4c4546', size:12}}, gridcolor:'rgba(0,0,0,0.06)', zerolinecolor:'rgba(0,0,0,0.12)' },
    shapes:[
      { type:'line', x0:d.userRisk, x1:d.userRisk, y0:0, y1:1, yref:'paper', line:{color:'rgba(186,26,26,0.28)', width:1, dash:'dash'} },
      { type:'line', x0:0, x1:1, xref:'paper', y0:d.userReturn, y1:d.userReturn, line:{color:'rgba(186,26,26,0.28)', width:1, dash:'dash'} },
    ],
    hoverlabel:{ bgcolor:'#ffffff', bordercolor:'#e5e5e5', font:{family:'Manrope', color:'#1a1c1c', size:11} },
  };

  Plotly.newPlot('chart', traces, layout, { displayModeBar:true, displaylogo:false, responsive:true, modeBarButtonsToRemove:['lasso2d','select2d'] });
}

function renderTopTable(d){
  let html = '<thead><tr><th class="stock">Stock</th><th>Risk (VaR95)</th><th>Avg Return</th><th>Distance</th></tr></thead><tbody>';
  d.topStocks.forEach(r=>{
    html += `<tr><td class="stock">${r.stock}</td><td>${fmt4(r.var95)}</td><td>${fmt4(r.avg_ret)}</td><td>${fmt4(r.dist)}</td></tr>`;
  });
  html += '</tbody>';
  document.getElementById('topTable').innerHTML = html;
}

function renderCorrCov(d){
  const corrTag = document.getElementById('corrTag');
  const covTag = document.getElementById('covTag');
  if(!d.enoughData){
    corrTag.textContent = 'insufficient overlapping periods';
    covTag.textContent = 'insufficient overlapping periods';
    document.getElementById('corrTable').innerHTML = '<tbody><tr><td style="padding:14px; color:#8c8a82; font-family:Manrope; font-size:12px;">Not enough overlapping price history across the selected stocks to compute correlation.</td></tr></tbody>';
    document.getElementById('covTable').innerHTML = '<tbody><tr><td style="padding:14px; color:#8c8a82; font-family:Manrope; font-size:12px;">Weights fell back to equal-weight allocation.</td></tr></tbody>';
    return;
  }
  corrTag.textContent = `${d.periods} periods`;
  covTag.textContent = `${d.periods} periods`;

  function buildHeatTable(matrix, formatter, absMax){
    let html = '<thead><tr><th></th>' + d.best.map(s=>`<th>${s}</th>`).join('') + '</tr></thead><tbody>';
    matrix.forEach((row,i)=>{
      html += `<tr><td class="stock">${d.best[i]}</td>` + row.map(v=>`<td class="heat-cell" style="${heatColor(v, absMax)}">${formatter(v)}</td>`).join('') + '</tr>';
    });
    html += '</tbody>';
    return html;
  }
  document.getElementById('corrTable').innerHTML = buildHeatTable(d.corrMatrix, fmt2, 1);
  const covAbsMax = Math.max(...d.covMatrix.flat().map(Math.abs), 1e-9);
  document.getElementById('covTable').innerHTML = buildHeatTable(d.covMatrix, fmt4, covAbsMax);
}

function renderWeights(d){
  let html = '<thead><tr><th class="stock">Stock</th><th>Weight</th></tr></thead><tbody>';
  d.best.forEach((s,i)=>{
    const w = d.optWeights[i];
    html += `<tr><td class="stock">${s}</td><td><div class="bar-cell"><span style="min-width:52px;">${fmt2(w*100)}%</span><div class="bar-track"><div class="bar-fill" style="width:${(w*100).toFixed(1)}%"></div></div></div></td></tr>`;
  });
  html += '</tbody>';
  document.getElementById('weightsTable').innerHTML = html;
}

function renderAllocation(d, totalAlloc){
  let html = '<thead><tr><th class="stock">Stock</th><th>Weight</th><th>Amount</th></tr></thead><tbody>';
  d.allocation.forEach(a=>{
    html += `<tr><td class="stock">${a.stock}</td><td><div class="bar-cell"><span style="min-width:52px;">${fmt2(a.weight*100)}%</span><div class="bar-track"><div class="bar-fill error" style="width:${(a.weight*100).toFixed(1)}%"></div></div></div></td><td>${fmtRupee(a.amount)}</td></tr>`;
  });
  html += '</tbody>';
  document.getElementById('allocTable').innerHTML = html;
  document.getElementById('allocNote').textContent = `Total allocated after round-up: ${fmtRupee(totalAlloc)} · requested ${fmtRupee(d.investAmount)}`;
}
</script>
</body>
</html>
"""


def fetch_one(stock, period_str):
    """Fetch one stock's daily closing price series. Returns (stock, series|None)."""
    ticker = f"{stock}.NS"
    try:
        df = yf.Ticker(ticker).history(period=period_str, interval="1d")
        if df.empty:
            return stock, None
        df.index = df.index.tz_localize(None)
        return stock, df["Close"]
    except Exception:
        return stock, None


@app.route("/")
def index():
    return INDEX_HTML


@app.route("/api/fetch", methods=["POST"])
def fetch_data():
    body = request.get_json(force=True) or {}
    years = int(body.get("years", 3))
    var_period = body.get("varPeriod", "monthly")
    return_period = body.get("returnPeriod", "monthly")

    if var_period not in ("monthly", "quarterly"):
        var_period = "monthly"
    if return_period not in ("monthly", "quarterly"):
        return_period = "monthly"
    years = max(1, years)

    period_str = f"{years}y"
    var_rule = "ME" if var_period == "monthly" else "QE"
    return_rule = "ME" if return_period == "monthly" else "QE"

    print(f"Fetching {len(STOCKS)} stocks, period={period_str} ...")
    all_data = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_one, s, period_str): s for s in STOCKS}
        for future in concurrent.futures.as_completed(futures):
            stock, series = future.result()
            if series is not None:
                all_data[stock] = series
            else:
                print(f"  failed/empty: {stock}")

    stock_rows = []
    risk_rows = []

    for stock, series in all_data.items():
        temp = series.reset_index()
        temp.columns = ["Date", "Close"]
        for _, r in temp.iterrows():
            stock_rows.append({
                "stock": stock,
                "Date": r["Date"].strftime("%Y-%m-%d"),
                "Close": float(r["Close"]),
            })

        s = series.dropna()
        var_close = s.resample(var_rule).last()
        var_returns = var_close.pct_change().dropna()
        ret_close = s.resample(return_rule).last()
        ret_returns = ret_close.pct_change().dropna()

        if len(var_returns) == 0 or len(ret_returns) == 0:
            continue

        avg_return = float(ret_returns.mean() * 100)
        var95 = float(np.percentile(var_returns * 100, 5))

        risk_rows.append({
            "stock": stock,
            "var95": var95,
            "avg_ret": avg_return,
            "var_period": var_period,
            "return_period": return_period,
        })

    print(f"Done. {len(risk_rows)} stocks with usable risk data, {len(stock_rows)} price rows.")
    return jsonify({"stockRows": stock_rows, "riskRows": risk_rows})


def open_browser():
    webbrowser.open(f"http://localhost:{PORT}/")


if __name__ == "__main__":
    threading.Timer(1.0, open_browser).start()
    app.run(port=PORT, debug=False)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\irfan alam\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\irfan alam\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\irfan alam\anaconda3\lib\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\irfan alam\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\irfan ala

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\irfan alam\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\irfan alam\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\irfan alam\anaconda3\lib\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\irfan alam\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\irfan ala

AttributeError: _ARRAY_API not found

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [02/Sep/2026 11:28:12] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Sep/2026 11:28:13] "GET /favicon.ico HTTP/1.1" 404 -


Fetching 99 stocks, period=3y ...


127.0.0.1 - - [02/Sep/2026 11:28:40] "POST /api/fetch HTTP/1.1" 200 -


Done. 99 stocks with usable risk data, 73531 price rows.
